# Project 1: Data Cleaning & Preparation

**Goal:** Clean a raw orders dataset by handling missing values, duplicates and incorrect data.

**To0ls:** Python, pandas, Jupyter Notebook

In [1]:
import pandas as pd

df = pd.read_excel(r"C:\Users\spectre\Desktop\Decodelabs\Dataset for Data Analytics - Copy.xlsx")

C:\Users\spectre\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
C:\Users\spectre\anaconda3\Lib\site-packages\pandas\core\arrays\masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (


In [2]:
df.head()

,OrderID,Date,CustomerID,Product,Quantity,UnitPrice,ShippingAddress,PaymentMethod,OrderStatus,TrackingNumber,ItemsInCart,CouponCode,ReferralSource,TotalPrice
0,ORD200000,2023-01-04,C72649,Monitor,5,570.62,928 Main St,Debit Card,Shipped,TRK37947903,7,SAVE10,Instagram,2853.10
1,ORD200001,2024-08-23,C75739,Phone,2,151.35,823 Main St,Online,Shipped,TRK91186779,3,SAVE10,Referral,302.70
2,ORD200002,2024-02-27,C81728,Tablet,5,550.68,512 Main St,Credit Card,Cancelled,TRK42903982,8,FREESHIP,Email,2753.40
3,ORD200003,2023-10-15,C33540,Chair,1,273.19,275 Main St,Debit Card,Returned,TRK62788070,5,SAVE10,Facebook,273.19
4,ORD200004,2025-05-08,C81840,Printer,4,626.01,668 Main St,Online,Delivered,TRK29241424,8,SAVE10,Email,2504.04


In [3]:
df.shape

(1200, 14)

In [4]:
df.info

<bound method DataFrame.info of         OrderID       Date CustomerID  Product  Quantity  UnitPrice  \
0     ORD200000 2023-01-04     C72649  Monitor         5     570.62   
1     ORD200001 2024-08-23     C75739    Phone         2     151.35   
2     ORD200002 2024-02-27     C81728   Tablet         5     550.68   
3     ORD200003 2023-10-15     C33540    Chair         1     273.19   
4     ORD200004 2025-05-08     C81840  Printer         4     626.01   
...         ...        ...        ...      ...       ...        ...   
1195  ORD201195 2024-06-20     C21126     Desk         1     107.04   
1196  ORD201196 2024-03-04     C20095  Monitor         2     662.53   
1197  ORD201197 2023-07-13     C79674   Tablet         2     436.84   
1198  ORD201198 2024-08-22     C64753    Chair         4     262.52   
1199  ORD201199 2023-06-11     C57502   Tablet         4     560.58   

     ShippingAddress PaymentMethod OrderStatus TrackingNumber  ItemsInCart  \
0        928 Main St    Debit Card   

## Making a copy

In [5]:
df_clean = df.copy()

## Missing values

In [6]:
df_clean.isnull().sum()

OrderID              0
Date                 0
CustomerID           0
Product              0
Quantity             0
UnitPrice            0
ShippingAddress      0
PaymentMethod        0
OrderStatus          0
TrackingNumber       0
ItemsInCart          0
CouponCode         309
ReferralSource       0
TotalPrice           0
dtype: int64

**Finding:** Only CouponCode had missing values (309)

**Decision:** A blank most likely meant no coupon, so I filled them with "No Coupon" instead of deleting them.

In [7]:
df_clean["CouponCode"] = df_clean["CouponCode"].fillna("No Coupon")

In [8]:
df_clean["CouponCode"].isnull().sum()

0

In [9]:
df_clean["CouponCode"].value_counts()

CouponCode
FREESHIP     313
No Coupon    309
WINTER15     292
SAVE10       286
Name: count, dtype: int64

In [10]:
df_clean["OrderID"].duplicated().sum()

0

**Finding:** No duplicate rows and no duplictate OrderIds.

**Decision:** Nothing to remove. Repeated CustomerIds are normal, since customer can order more than once.

In [11]:
df_clean = df_clean.drop_duplicates()

In [12]:
len(df_clean)

1200

In [13]:
df_clean.dtypes

OrderID                       str
Date               datetime64[us]
CustomerID                    str
Product                       str
Quantity                    int64
UnitPrice                 float64
ShippingAddress               str
PaymentMethod                 str
OrderStatus                   str
TrackingNumber                str
ItemsInCart                 int64
CouponCode                    str
ReferralSource                str
TotalPrice                float64
dtype: object

In [14]:
df_clean["Date"] = pd.to_datetime(df_clean["Date"])

In [15]:
num_cols = ["Quantity", "UnitPrice", "ItemsInCart", "TotalPrice"]
for col in num_cols:
    df_clean[col] = pd.to_numeric(df_clean[col], errors="coerce")

In [16]:
text_cols = ["OrderID", "CustomerID", "Product", "ShippingAddress",
             "PaymentMethod", "OrderStatus", "TrackingNumber",
             "CouponCode", "ReferralSource"]
for col in text_cols:
    df_clean[col] = df_clean[col].str.strip()

In [17]:
df_clean["UnitPrice"] = df_clean["UnitPrice"].round(2)
df_clean["TotalPrice"] = df_clean["TotalPrice"].round(2)

In [18]:
df_clean[["Quantity", "UnitPrice", "ItemsInCart", "TotalPrice"]].describe()

,Quantity,UnitPrice,ItemsInCart,TotalPrice
count,1200.000000,1200.000000,1200.000000,1200.000000
mean,2.945833,356.412750,5.485000,1053.968300
std,1.407557,197.177146,2.281983,819.856558
min,1.000000,11.390000,1.000000,11.390000
25%,2.000000,186.062500,4.000000,410.520000
50%,3.000000,364.210000,5.000000,823.615000
75%,4.000000,521.570000,7.000000,1578.475000
max,5.000000,699.930000,10.000000,3456.400000


In [19]:
(df_clean["Quantity"] <= 0).sum()
(df_clean["UnitPrice"] <= 0). sum()

0

In [20]:
calc = (df_clean["Quantity"] * df_clean["UnitPrice"]).round(2)
mismatch = (calc != df_clean["TotalPrice"]).sum()
mismatch

0

**Finding:** Dates were already real dates and numbers were stored as numbers.

**Decision:** trimmed soaces in text and rounded prices to 2 decimal place.

In [21]:
for col in ["Product", "PaymentMethod", "OrderStatus", "ReferralSource"]:
    print(df_clean[col].value_counts())
    print()

Product
Printer    181
Tablet     179
Chair      178
Laptop     173
Desk       170
Monitor    163
Phone      156
Name: count, dtype: int64

PaymentMethod
Online         258
Cash           246
Credit Card    234
Debit Card     232
Gift Card      230
Name: count, dtype: int64

OrderStatus
Cancelled    250
Returned     247
Pending      237
Shipped      235
Delivered    231
Name: count, dtype: int64

ReferralSource
Instagram    259
Email        250
Google       241
Facebook     228
Referral     222
Name: count, dtype: int64



In [22]:
df_clean.to_excel("Cleaned_Data.xlsx", index=False)

In [23]:
log = pd.DataFrame({
    "Issue": ["Missing values", "Duplicates", "Wrong date formats", "Numbers sorted as text"],
    "Column": ["CouponCode", "All column", "Date", "Quantity, UnitPrice, ItemsInCart, TotalPrice"],
    "Found": [309,0,0,0],
    "Action": ["Filled with 'No Coupon'", "None needed", "All dates are real dates", "None found"]
})
log

,Issue,Column,Found,Action
0,Missing values,CouponCode,309,Filled with 'No Coupon'
1,Duplicates,All column,0,None needed
2,Wrong date formats,Date,0,All dates are real dates
3,Numbers sorted as text,"Quantity, UnitPrice, ItemsInCart, TotalPrice",0,None found
